<a href="https://colab.research.google.com/github/edgaralbasanz/ealba-xatbot/blob/main/XatBot2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import userdata
import google.generativeai as genai
import requests
from bs4 import BeautifulSoup
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok
import re
from urllib.parse import urljoin, urlparse
import time

# =================== SECRETS ===================
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
genai.configure(api_key=GEMINI_API_KEY)

# =================== MODEL (millor quota) ===================
model = genai.GenerativeModel('gemini-2.5-flash-lite')

app = Flask(__name__)
CORS(app)

# =================== CRAWLER SENSE LÍMIT ===================
print("🚀 Iniciant CRAWLER COMPLET SENSE LÍMIT...")

BASE_URL = "https://ealba.inscastellbisbal.net"
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}

visited = set()
to_visit = [BASE_URL]
all_content = []

start_time = time.time()
max_time = 180  # 3 minuts màxim

while to_visit and (time.time() - start_time) < max_time:
    current_url = to_visit.pop(0)
    if current_url in visited:
        continue

    print(f"   📄 Scraping: {current_url}")
    visited.add(current_url)

    try:
        r = requests.get(current_url, headers=headers, timeout=10)
        if r.status_code != 200:
            continue

        soup = BeautifulSoup(r.text, 'html.parser')

        texts = []
        for tag in soup.find_all(['h1','h2','h3','h4','p','li','strong','em','blockquote','div']):
            text = tag.get_text(strip=True)
            if len(text) > 30:
                texts.append(text)

        for img in soup.find_all('img'):
            alt = img.get('alt', '').strip()
            if alt and len(alt) > 15:
                texts.append(f"Imatge: {alt}")

        if texts:
            all_content.append(" | ".join(texts))

        for link in soup.find_all('a', href=True):
            full_url = urljoin(current_url, link['href'])
            if urlparse(full_url).netloc in ["", "ealba.inscastellbisbal.net"]:
                if full_url not in visited and full_url not in to_visit:
                    if not full_url.endswith(('.pdf','.jpg','.png','.zip','.doc')):
                        to_visit.append(full_url)
    except:
        continue

    time.sleep(0.4)

# =================== CONTEXT GLOBAL ===================
GLOBAL_CONTEXT = " ".join(all_content)
GLOBAL_CONTEXT = re.sub(r'\s+', ' ', GLOBAL_CONTEXT)

print(f"\n✅ CRAWLER FINALITZAT!")
print(f"   Pàgines visitades: {len(visited)}")
print(f"   Caràcters: {len(GLOBAL_CONTEXT)}")

# =================== Neteja resposta ===================
def clean_response(text):
    text = re.sub(r'\*\*(.+?)\*\*', r'\1', text)
    text = re.sub(r'\*(.+?)\*', r'\1', text)
    lines = text.split('\n')
    if len(lines) > 14:
        text = '\n'.join(lines[:14]) + "\n\n..."
    return text.strip()

# =================== CHAT ===================
@app.route('/chat', methods=['POST'])
def chat():
    try:
        user_message = request.json.get('message', '') if request.json else ''

        prompt = f"""Ets l'assistent oficial de Sergi Villanueva.
Respon de forma clara, directa i concisa (màxim 10-12 línies).
Utilitza paràgrafs curts.
No utilitzis Markdown (** o *).
Sigues professional però natural.

Informació completa del lloc web:
{GLOBAL_CONTEXT[:17000]}

Pregunta: {user_message}"""

        response = model.generate_content(prompt)
        clean_reply = clean_response(response.text)

        return jsonify({"reply": clean_reply})

    except Exception as e:
        print(f"Error: {e}")
        return jsonify({"reply": "Ho sento, hi ha hagut un error. Torna-ho a provar."})

# ===================== INICIAR =====================
public_url = ngrok.connect(5000)
print("\n" + "="*85)
print("✅ XATBOT FINAL - CRAWLER SENSE LÍMIT + RESPOSTES CURTES")
print(f"🔗 URL: {public_url}")
print(f"📊 Pàgines carregades: {len(visited)}")
print("="*85)

app.run(port=5000)

🚀 Iniciant CRAWLER COMPLET SENSE LÍMIT...
   📄 Scraping: https://ealba.inscastellbisbal.net
   📄 Scraping: https://ealba.inscastellbisbal.net#wp--skip-link--target
   📄 Scraping: https://ealba.inscastellbisbal.net/
   📄 Scraping: https://ealba.inscastellbisbal.net/apunts/
   📄 Scraping: https://ealba.inscastellbisbal.net/politica-de-privadesa/
   📄 Scraping: https://ealba.inscastellbisbal.net/politica-de-privadesa/privacy-policy/
   📄 Scraping: https://ealba.inscastellbisbal.net/reptes/
   📄 Scraping: https://ealba.inscastellbisbal.net/reptes/repte-1-1/
   📄 Scraping: https://ealba.inscastellbisbal.net/reptes/repte-1-1/espai-de-treball-al-nuvol/
   📄 Scraping: https://ealba.inscastellbisbal.net/reptes/repte-1-1/correu-professional/
   📄 Scraping: https://ealba.inscastellbisbal.net/reptes/repte-1-1/correu-corporatiu-del-centre/
   📄 Scraping: https://ealba.inscastellbisbal.net/reptes/repte-1-1/portafolis-digital/
   📄 Scraping: https://ealba.inscastellbisbal.net/reptes/repte-1-1/platafo

INFO:werkzeug:127.0.0.1 - - [08/Jun/2026 09:57:25] "POST /chat HTTP/1.1" 200 -


Error: HTTPConnectionPool(host='localhost', port=35175): Read timed out. (read timeout=600.0)
   📄 Scraping: https://ealba.inscastellbisbal.net/index.php/utilitzacio-de-marcadors-en-documents-de-drive/
   📄 Scraping: https://ealba.inscastellbisbal.net/index.php/comparativa-de-gestors-de-fitxers-al-nuvol/
   📄 Scraping: https://ealba.inscastellbisbal.net/index.php/copies-de-seguretat-al-nuvol/
   📄 Scraping: https://ealba.inscastellbisbal.net/index.php/sincronitzacio-descriptori-amb-el-nuvol/
   📄 Scraping: https://ealba.inscastellbisbal.net/reptes/repte-1-1/correu-professional/#wp--skip-link--target
   📄 Scraping: https://ealba.inscastellbisbal.net/index.php/creacio-dun-correu-per-temes-academics-i-professionals-identificatiu/
   📄 Scraping: https://ealba.inscastellbisbal.net/index.php/configuracio-duna-signatura/
   📄 Scraping: https://ealba.inscastellbisbal.net/index.php/configuracio-de-redireccio-al-correu-corporatiu/
   📄 Scraping: https://ealba.inscastellbisbal.net/index.php/poder

INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
